In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install konlpy
!pip install soynlp
!pip instlal sentenecepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 59.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 114.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 90.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 40.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.3/776.3 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 87.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 7.3 MB/s eta 0:00:00
ERROR: unknown command "instlal" - maybe you meant "install"


In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60
    fold = 5

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_train_pre.csv', encoding = 'utf-8')
df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test_pre.csv', encoding = 'utf-8')

df_train_pre.dropna(inplace = True)
df_dev_pre.dropna(inplace = True)
df_test_pre.dropna(inplace = True)

df_train_pre.reset_index(drop = True, inplace = True)
df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [ ]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = 'EleutherAI/polyglot-ko-1.3b'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True)
tokenizer.pad_token = tokenizer.eos_token

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch])  # 레이블을 Tensor로 변환

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks, labels

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

df = pd.concat([df_train_pre, df_dev_pre])

# KFold 교차 검증
fold = 0
for fold, (train_index, valid_index) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}/{args.fold}")

    # 데이터셋 분할
    df_train = df.iloc[train_index].reset_index(drop=True)
    df_valid = df.iloc[valid_index].reset_index(drop=True)

    # DataLoader 설정
    train_dataset = CustomDataset(df_train, 'input', 'output', tokenizer, args.max_len)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)

    valid_dataset = CustomDataset(df_valid, 'input', 'output', tokenizer, args.max_len)
    valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

    # 모델 초기화 및 학습
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size, output_attentions=False, output_hidden_states=False)
    # GPTNeoXForSequenceClassification.from_pretrained로 하면 결과가 달라지려나..?
    model.to(device)
    model.config.pad_token_id = tokenizer.pad_token_id

    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=args.start_lr)
    scheduler = get_linear_schedule_with_warmup(optimizer,
                                                num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                                num_training_steps = len(train_loader)*args.epochs)

    stop_val_f1 = 0
    stop_count = 0

    for epoch in range(args.epochs):
        if stop_count == 3:
            break
        model.train()
        train_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(train_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            outputs = model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()
            scheduler.step()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / len(pbar), 4)))

        train_loss = train_loss / len(train_loader)
        train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
        train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        model.eval()
        valid_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(valid_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            outputs = model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            valid_loss += loss.item()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / len(pbar), 4)))
        valid_loss = valid_loss / len(valid_loader)
        valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
        valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        print(f'Fold {fold + 1}, Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
        print(f'                     v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

        if valid_loss > stop_val_f1:
            default_weight_path = path + f'fold{fold + 1}_polyglot_ko_kfold.pt'
            torch.save(model.state_dict(), default_weight_path)
            stop_val_loss = valid_loss
        else:
            stop_count += 1

        torch.cuda.empty_cache()
        gc.collect()

Fold 1/5


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-1.3b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
[C_loss : 0.2008]: 100%|██████████| 116/116 [00:35<00:00,  3.27it/s]


Fold 1, Epoch : 1,    t_loss : 0.2801,   t_f1 : 0.8737,   t_acc : 0.8792

                                v_loss : 0.2008,   v_f1 : 0.9186,   v_acc : 0.923



[C_loss : 0.2538]: 100%|██████████| 116/116 [00:35<00:00,  3.27it/s]


...

Fold 5, Epoch : 8,    t_loss : 0.0044,   t_f1 : 0.9993,   t_acc : 0.9993

                                v_loss : 0.3489,   v_f1 : 0.927,   v_acc : 0.93



[C_loss : 0.4006]: 100%|██████████| 116/116 [00:35<00:00,  3.27it/s]


Fold 5, Epoch : 9,    t_loss : 0.0004,   t_f1 : 0.9999,   t_acc : 0.9999

                                v_loss : 0.4006,   v_f1 : 0.9277,   v_acc : 0.9308



[C_loss : 0.4078]: 100%|██████████| 116/116 [00:35<00:00,  3.27it/s]


Fold 5, Epoch : 10,    t_loss : 0.0003,   t_f1 : 0.9999,   t_acc : 0.9999

                                v_loss : 0.4078,   v_f1 : 0.9281,   v_acc : 0.9311



In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks

model_name = 'EleutherAI/polyglot-ko-1.3b'

tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True)
tokenizer.pad_token = tokenizer.eos_token
test_dataset = TestDataset(df_test_pre, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size, output_attentions=False, output_hidden_states=False)
# GPTNeoXForSequenceClassification.from_pretrained로 하면 결과가 달라지려나..?
model.to(device)
model.config.pad_token_id = tokenizer.pad_token_id

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-1.3b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from collections import Counter

model_paths = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_polyglot_ko_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_polyglot_ko_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_polyglot_ko_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_polyglot_ko_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_polyglot_ko_kfold.pt']
models = []
for model_path in model_paths:
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    models.append(model)

res = []
with torch.no_grad():
    for _, data in enumerate(tqdm(test_loader)):
        ids = data[0].to(device)
        mask = data[1].to(device)

        outputs = [model(ids, mask)[0] for model in models]

        avg_output = sum(outputs) / len(models)
        avg_output = avg_output.cpu().numpy()

        res.extend(avg_output.argmax(axis=1).tolist())

df_test['output'] = res

100%|██████████| 65/65 [01:36<00:00,  1.49s/it]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/test4_polyglot_ko_2.jsonl')